In [1]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# ---------------------------------------------
# (a) Generate motor predictive maintenance data
# ---------------------------------------------
np.random.seed(0)

n = 1000

operating_hours = np.random.uniform(0, 20000, n)
ambient_temp = np.random.uniform(20, 45, n)
load_percent = np.random.uniform(20, 100, n)
supply_voltage = np.random.uniform(210, 240, n)
vibration_rms = np.random.uniform(0.1, 5.0, n)
oil_viscosity = np.random.uniform(10, 100, n)

# ---------------------------------------------
# Create DataFrame
# ---------------------------------------------
df = pd.DataFrame({
    "operating_hours": operating_hours,
    "ambient_temp": ambient_temp,
    "load_percent": load_percent,
    "supply_voltage": supply_voltage,
    "vibration_rms": vibration_rms,
    "oil_viscosity": oil_viscosity
})

# ---------------------------------------------
# (b) Feature engineering
# ---------------------------------------------
df["power_factor"] = df["load_percent"] / df["supply_voltage"]

df["heat_index"] = (
    df["ambient_temp"] * df["load_percent"] / 100
)

df["wear_rate"] = (
    df["operating_hours"] * df["vibration_rms"]
)

# ---------------------------------------------
# Target: time_to_failure
# Lower with higher load/vibration/wear
# ---------------------------------------------
noise = np.random.normal(0, 200, n)

time_to_failure = (
    20000
    - 30 * load_percent
    - 1500 * vibration_rms
    - 0.03 * operating_hours
    - 2 * ambient_temp
    - 0.0005 * df["wear_rate"]
    + 5 * oil_viscosity
    + noise
)

# ---------------------------------------------
# Features and target
# ---------------------------------------------
X = df

y = time_to_failure

# ---------------------------------------------
# Train/test split
# ---------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ---------------------------------------------
# (c) Build models
# ---------------------------------------------
models = {
    "LinearRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "PolynomialDegree2": Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2)),
        ("model", LinearRegression())
    ]),

    "RidgeRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])
}

# ---------------------------------------------
# (d) 5-Fold Cross Validation
# ---------------------------------------------
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = {}

print("MODEL COMPARISON")
print("-" * 50)

for name, pipeline in models.items():

    # R2 scores
    r2_scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring="r2"
    )

    # RMSE scores
    rmse_scores = np.sqrt(
        -cross_val_score(
            pipeline,
            X,
            y,
            cv=cv,
            scoring="neg_mean_squared_error"
        )
    )

    results[name] = {
        "mean_r2": r2_scores.mean(),
        "std_r2": r2_scores.std(),
        "mean_rmse": rmse_scores.mean(),
        "std_rmse": rmse_scores.std()
    }

    print(f"\n{name}")
    print(f"R2:   {r2_scores.mean():.4f} ± {r2_scores.std():.4f}")
    print(f"RMSE: {rmse_scores.mean():.4f} ± {rmse_scores.std():.4f}")

# ---------------------------------------------
# Select best model based on R2
# ---------------------------------------------
best_model_name = max(
    results,
    key=lambda x: results[x]["mean_r2"]
)

best_pipeline = models[best_model_name]

print("\n" + "=" * 50)
print(f"Best Model: {best_model_name}")

# ---------------------------------------------
# Train best model on full training data
# ---------------------------------------------
best_pipeline.fit(X_train, y_train)

# Evaluate on test set
y_pred = best_pipeline.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test R2:   {test_r2:.4f}")

# ---------------------------------------------
# (e) Save best model pipeline
# ---------------------------------------------
joblib.dump(best_pipeline, "best_predictive_maintenance_model.pkl")

print("\nModel saved successfully!")
print("Filename: best_predictive_maintenance_model.pkl")

MODEL COMPARISON
--------------------------------------------------

LinearRegression
R2:   0.9915 ± 0.0006
RMSE: 200.9506 ± 3.4725

PolynomialDegree2
R2:   0.9907 ± 0.0009
RMSE: 210.3529 ± 6.7314

RidgeRegression
R2:   0.9915 ± 0.0006
RMSE: 201.0558 ± 3.8989

Best Model: LinearRegression
Test RMSE: 195.5442
Test R2:   0.9921

Model saved successfully!
Filename: best_predictive_maintenance_model.pkl
